In [5]:
import pandas as pd

df = pd.read_csv("data.csv")
events = pd.read_csv("events.csv")

date_cols = [
    "user_id",
    "product_id",
    "order_id",
    "created_at",
    "shipped_at",
    "delivered_at",
    "returned_at",
    "sold_at"
]

for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], utc=True, format="ISO8601", errors="coerce")

df["returned"] = df["returned_at"].notna().astype(int)

In [16]:
print(df["created_at"][0], df["created_at"][len(df["created_at"]) - 1])

2026-03-11 00:47:27+00:00 2019-01-04 03:52:40+00:00


In [6]:
df[["created_at", "delivered_at"]].head()

,created_at,delivered_at
0,2026-03-11 00:47:27+00:00,NaT
1,2026-03-11 00:46:02+00:00,NaT
2,2026-03-11 00:46:02+00:00,NaT
3,2026-03-11 00:39:44.268552+00:00,2026-03-16 01:00:44.268552+00:00
4,2026-03-11 00:39:29.765939+00:00,NaT


In [10]:
# delivery time
df["delivery_time"] = (
    df["delivered_at"] - df["created_at"]
).dt.total_seconds() / 3600

## Проверка влияния времени доставки на возвраты

In [11]:
# базовая статистика
df["returned"].mean(), df.shape

(np.float64(0.09926013873772853), (545778, 51))

In [12]:
# сравнение возвратов и не-возвратов
df.groupby("returned")["delivery_time"].describe()

,count,mean,std,min,25%,50%,75%,max
returned,,,,,,,,
0,135951.0,95.980943,40.344374,0.883333,65.633333,95.916667,125.983333,191.550000
1,54174.0,95.929432,40.537724,1.083333,65.466667,95.933333,126.350000,191.233333


In [13]:
# проверка статистической разницы
from scipy.stats import mannwhitneyu

a = df[df["returned"] == 0]["delivery_time"].dropna()
b = df[df["returned"] == 1]["delivery_time"].dropna()

mannwhitneyu(a, b, alternative="two-sided")

MannwhitneyuResult(statistic=np.float64(3683524500.0), pvalue=np.float64(0.924789644113454))

Доля возвратов 9,9 %, разницы по времени доставки между возвратами и не возвратами нет
Стат различий нет
Время доставки не объясняет возвраты. Поведение возвратов определяется другими факторами (вероятнее всего — цена, скидка, товарная категория или клиентский профиль).

Скидок на самом деле нет

In [14]:
df = pd.read_csv("data.csv")
print(df["sale_price"].isna().sum())
print(df["product_retail_price"].isna().sum())

print((df["product_retail_price"] - df["sale_price"]).describe())

print((df["product_retail_price"] > df["sale_price"]).sum())
print((df["product_retail_price"] == df["sale_price"]).sum())
print((df["product_retail_price"] < df["sale_price"]).sum())


0
0
count    545778.0
mean          0.0
std           0.0
min           0.0
25%           0.0
50%           0.0
75%           0.0
max           0.0
dtype: float64
0
545778
0
